# Time-Series Sales Data Preprocessing (Pre-EDA)

This notebook executes the foundational data preprocessing steps prior to Exploratory Data Analysis (EDA) for hourly sales forecasting.

### Pipeline Overview
1. **Load the Dataset**: Read raw sales CSV and inspect shape & schema.
2. **Identify Time Column & Target**: Map timestamp and target sales variables.
3. **Convert Time Column to Datetime**: Standardize `datum` to `pd.DatetimeIndex`.
4. **Sort Chronologically**: Ensure chronological time order.
5. **Check Duplicate Timestamps**: Detect redundant timestamps and rows.
6. **Check Missing Values**: Audit missing entries across targets and timestamps.
7. **Check Data Types**: Validate data types for numerical, datetime, and categorical fields.
8. **Check Invalid Values**: Inspect negative values, zero sales patterns, and range boundaries.
9. **Check Time Frequency**: Verify regular time intervals (hourly diffs).
10. **Set Datetime Index & Export**: Set datetime index and export `dataset/saleshourly_preprocessed.csv`.


In [1]:
!pip install pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install numpy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Load the Dataset
Read the CSV file into a pandas DataFrame and inspect its initial shape, column names, and sample rows.


## Step 2: Identify Time Column and Target Variables
- **Time Column**: `datum` (contains timestamp strings e.g. `1/2/2014 8:00`)
- **Target Variables**: 8 drug group sales quantities (`M01AB`, `M01AE`, `N02BA`, `N02BE`, `N05B`, `N05C`, `R03`, `R06`)
- **Metadata Features**: `Year`, `Month`, `Hour`, `Weekday Name`


In [3]:
import os
import pandas as pd
import numpy as np

csv_path = '../dataset/saleshourly.csv' 
df = pd.read_csv(csv_path)

print(f"Successfully loaded dataset from: '{csv_path}'")
print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

Successfully loaded dataset from: '../dataset/saleshourly.csv'
Dataset Shape: (50532, 13)

Columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014 8:00,0.0,0.67,0.4,2.0,0.0,0.0,0.0,1.0,2014,1,8,Thursday
1,1/2/2014 9:00,0.0,0.00,1.0,0.0,2.0,0.0,0.0,0.0,2014,1,9,Thursday
2,1/2/2014 10:00,0.0,0.00,0.0,3.0,2.0,0.0,0.0,0.0,2014,1,10,Thursday
3,1/2/2014 11:00,0.0,0.00,0.0,2.0,1.0,0.0,0.0,0.0,2014,1,11,Thursday
4,1/2/2014 12:00,0.0,2.00,0.0,5.0,2.0,0.0,0.0,0.0,2014,1,12,Thursday


In [4]:
time_col = 'datum'
target_cols = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
feature_cols = ['Year', 'Month', 'Hour', 'Weekday Name']

print(f"Time column: {time_col}")
print(f"Target columns ({len(target_cols)}): {target_cols}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")


Time column: datum
Target columns (8): ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
Feature columns (4): ['Year', 'Month', 'Hour', 'Weekday Name']


## Step 3: Convert Time Column to Datetime
Parse the string representation of date and time into a standardized `datetime64` data type.


In [5]:
df['datetime'] = pd.to_datetime(df['datum'])

print("Datetime conversion sample:")
print(df[['datum', 'datetime']].head())
print(f"\nTime Range: {df['datetime'].min()} to {df['datetime'].max()}")


Datetime conversion sample:
            datum            datetime
0   1/2/2014 8:00 2014-01-02 08:00:00
1   1/2/2014 9:00 2014-01-02 09:00:00
2  1/2/2014 10:00 2014-01-02 10:00:00
3  1/2/2014 11:00 2014-01-02 11:00:00
4  1/2/2014 12:00 2014-01-02 12:00:00

Time Range: 2014-01-02 08:00:00 to 2019-10-08 19:00:00


## Step 4: Sort Chronologically
Ensure that all records are strictly ordered in ascending chronological order by `datetime`.


In [6]:
df = df.sort_values('datetime').reset_index(drop=True)

print("First 3 records (chronological):")
print(df[['datetime']].head(3))

print("\nLast 3 records (chronological):")
print(df[['datetime']].tail(3))


First 3 records (chronological):
             datetime
0 2014-01-02 08:00:00
1 2014-01-02 09:00:00
2 2014-01-02 10:00:00

Last 3 records (chronological):
                 datetime
50529 2019-10-08 17:00:00
50530 2019-10-08 18:00:00
50531 2019-10-08 19:00:00


## Step 5: Check Duplicate Timestamps and Rows
Verify whether there are duplicate timestamps or identical row entries.


In [7]:
duplicate_rows = df.duplicated().sum()
duplicate_timestamps = df['datetime'].duplicated().sum()

print(f"Duplicate full rows: {duplicate_rows}")
print(f"Duplicate timestamps: {duplicate_timestamps}")

if duplicate_timestamps == 0:
    print("STATUS: Every timestamp in the dataset is unique. No deduplication required.")
else:
    print("STATUS: Duplicates detected. Action needed.")


Duplicate full rows: 0
Duplicate timestamps: 0
STATUS: Every timestamp in the dataset is unique. No deduplication required.


## Step 6: Check Missing Values
Examine the dataset for missing (NaN/null) values in target sales columns and time features.


In [8]:
missing_counts = df.isnull().sum()
print("Missing values count per column:")
print(missing_counts)
print(f"\nTotal missing values across dataset: {missing_counts.sum()}")


Missing values count per column:
datum           0
M01AB           0
M01AE           0
N02BA           0
N02BE           0
N05B            0
N05C            0
R03             0
R06             0
Year            0
Month           0
Hour            0
Weekday Name    0
datetime        0
dtype: int64

Total missing values across dataset: 0


## Step 7: Check Data Types
Verify that target columns are floating point numbers, dates are `datetime64[ns]`, temporal indices are integers, and weekday names are categorical.


In [9]:
print("Data types before explicit casting:")
print(df.dtypes)

# Convert Weekday Name to categorical type
df['Weekday Name'] = df['Weekday Name'].astype('category')

print("\nUpdated Data Types:")
print(df.dtypes)


Data types before explicit casting:
datum                   object
M01AB                  float64
M01AE                  float64
N02BA                  float64
N02BE                  float64
N05B                   float64
N05C                   float64
R03                    float64
R06                    float64
Year                     int64
Month                    int64
Hour                     int64
Weekday Name            object
datetime        datetime64[ns]
dtype: object

Updated Data Types:
datum                   object
M01AB                  float64
M01AE                  float64
N02BA                  float64
N02BE                  float64
N05B                   float64
N05C                   float64
R03                    float64
R06                    float64
Year                     int64
Month                    int64
Hour                     int64
Weekday Name          category
datetime        datetime64[ns]
dtype: object


## Step 8: Check Invalid Values & Domain Integrity
Perform data validation checks:
1. **Negative sales**: Sales values cannot be negative.
2. **Zero sales distribution**: Analyze zero sales frequency (common in hourly sales data).
3. **Summary statistics**: Range and quantile distributions for all 8 drug groups.


In [10]:
# 1. Negative sales check
negative_counts = (df[target_cols] < 0).sum()
print("Negative sales count per product category:")
print(negative_counts)

# 2. Zero sales check
zero_counts = (df[target_cols] == 0).sum()
zero_pcts = (df[target_cols] == 0).mean() * 100

zero_summary = pd.DataFrame({
    'Zero Count': zero_counts,
    'Zero Percentage (%)': zero_pcts.round(2)
})
print("\nZero Sales Distribution:")
print(zero_summary)

# 3. Summary statistics
print("\nSummary Statistics of Sales Targets:")
print(df[target_cols].describe().T)


Negative sales count per product category:
M01AB    0
M01AE    0
N02BA    0
N02BE    0
N05B     0
N05C     0
R03      0
R06      0
dtype: int64

Zero Sales Distribution:
       Zero Count  Zero Percentage (%)
M01AB       41633                82.39
M01AE       41017                81.17
N02BA       43086                85.26
N02BE       30130                59.63
N05B        39974                79.11
N05C        49668                98.29
R03         47120                93.25
R06         45247                89.54

Summary Statistics of Sales Targets:
         count      mean       std  min  25%  50%    75%   max
M01AB  50532.0  0.209787  0.556003  0.0  0.0  0.0  0.000   7.0
M01AE  50532.0  0.162365  0.416109  0.0  0.0  0.0  0.000   6.0
N02BA  50532.0  0.161723  0.453211  0.0  0.0  0.0  0.000   6.5
N02BE  50532.0  1.246842  2.387392  0.0  0.0  0.0  1.875  29.0
N05B   50532.0  0.368989  0.930934  0.0  0.0  0.0  0.000  15.0
N05C   50532.0  0.024736  0.217871  0.0  0.0  0.0  0.000   6.0


## Step 9: Check Time Frequency
Calculate difference between consecutive timestamps to confirm regular hourly intervals without temporal gaps.


In [11]:
time_diffs = df['datetime'].diff().value_counts()
print("Frequency distribution of consecutive timestamp differences:")
print(time_diffs)

if len(time_diffs.dropna()) == 1 and time_diffs.index[0] == pd.Timedelta(hours=1):
    print("\nSTATUS: Perfectly continuous hourly time-series (1-hour step size throughout).")


Frequency distribution of consecutive timestamp differences:
datetime
0 days 01:00:00    50531
Name: count, dtype: int64

STATUS: Perfectly continuous hourly time-series (1-hour step size throughout).


C:\Users\ranje\AppData\Local\Temp\ipykernel_18308\4050628114.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  if len(time_diffs.dropna()) == 1 and time_diffs.index[0] == pd.Timedelta(hours=1):


## Step 10: Set Datetime Index and Export Preprocessed Dataset
Set `datetime` as the index of the DataFrame and export the cleaned data to `dataset/saleshourly_preprocessed.csv`.


In [12]:
import os

df_preprocessed = df.set_index('datetime')

output_dir = 'dataset' if os.path.exists('dataset') else '../dataset'
output_path = os.path.join(output_dir, 'saleshourly_preprocessed.csv')
df_preprocessed.to_csv(output_path)

print(f"Preprocessed dataset successfully saved to: '{output_path}'")
print("Preprocessed DataFrame Shape:", df_preprocessed.shape)
print("Index Name:", df_preprocessed.index.name)
df_preprocessed.head()

Preprocessed dataset successfully saved to: '../dataset\saleshourly_preprocessed.csv'
Preprocessed DataFrame Shape: (50532, 13)
Index Name: datetime


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
datetime,,,,,,,,,,,,,
2014-01-02 08:00:00,1/2/2014 8:00,0.0,0.67,0.4,2.0,0.0,0.0,0.0,1.0,2014,1,8,Thursday
2014-01-02 09:00:00,1/2/2014 9:00,0.0,0.00,1.0,0.0,2.0,0.0,0.0,0.0,2014,1,9,Thursday
2014-01-02 10:00:00,1/2/2014 10:00,0.0,0.00,0.0,3.0,2.0,0.0,0.0,0.0,2014,1,10,Thursday
2014-01-02 11:00:00,1/2/2014 11:00,0.0,0.00,0.0,2.0,1.0,0.0,0.0,0.0,2014,1,11,Thursday
2014-01-02 12:00:00,1/2/2014 12:00,0.0,2.00,0.0,5.0,2.0,0.0,0.0,0.0,2014,1,12,Thursday


## Step 11: Temporal Train, Validation, and Test Dataset Partitioning

Per the modeling cross-validation strategy:
- **Train Set (2014–2017)**: 4 full calendar years (`2014-01-02 08:00:00` to `2017-12-31 23:00:00`) for model parameter estimation.
- **Validation Set (2018)**: 1 full calendar year (`2018-01-01 00:00:00` to `2018-12-31 23:00:00`) for hyperparameter tuning and model selection.
- **Test Set (2019)**: Holdout evaluation period (`2019-01-01 00:00:00` to `2019-10-08 19:00:00`) for final model benchmarking.

We export both **Hourly** and **Daily Aggregated** versions of each split into the `dataset/` directory.

In [13]:
import os
import pandas as pd

# Load preprocessed data if needed
DATA_PATH = '../dataset/saleshourly_preprocessed.csv' if os.path.exists('../dataset/saleshourly_preprocessed.csv') else 'dataset/saleshourly_preprocessed.csv'
df_p = pd.read_csv(DATA_PATH)
df_p['datetime'] = pd.to_datetime(df_p['datetime'])

# 1. Hourly Splits
train_h = df_p[df_p['Year'].isin([2014, 2015, 2016, 2017])].copy()
val_h = df_p[df_p['Year'] == 2018].copy()
test_h = df_p[df_p['Year'] == 2019].copy()

output_dir = '../dataset' if os.path.exists('../dataset') else 'dataset'

train_h.to_csv(os.path.join(output_dir, 'train_hourly.csv'), index=False)
val_h.to_csv(os.path.join(output_dir, 'val_hourly.csv'), index=False)
test_h.to_csv(os.path.join(output_dir, 'test_hourly.csv'), index=False)

train_h.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
val_h.to_csv(os.path.join(output_dir, 'val.csv'), index=False)
test_h.to_csv(os.path.join(output_dir, 'test.csv'), index=False)

print(f"Hourly Train (2014-2017): {len(train_h):,} rows ({train_h['datetime'].min()} to {train_h['datetime'].max()})")
print(f"Hourly Val   (2018):      {len(val_h):,} rows ({val_h['datetime'].min()} to {val_h['datetime'].max()})")
print(f"Hourly Test  (2019):      {len(test_h):,} rows ({test_h['datetime'].min()} to {test_h['datetime'].max()})")

Hourly Train (2014-2017): 35,032 rows (2014-01-02 08:00:00 to 2017-12-31 23:00:00)
Hourly Val   (2018):      8,760 rows (2018-01-01 00:00:00 to 2018-12-31 23:00:00)
Hourly Test  (2019):      6,740 rows (2019-01-01 00:00:00 to 2019-10-08 19:00:00)
